[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)][def]

[def]: https://colab.research.google.com/github/LEE-Sanghoon/ITE4052/blob/main/02.Region-Level_Processing.ipynb

# 영역기반 영상처리

### 흐림 효과

- 해당 픽셀의 주변 값들과 비교하여 계산하고, 픽셀의 값을 조정
- 블러링(blurring) 또는 스무딩(smoothing)

In [ ]:
import cv2

img_file = "Lena.png"
src = cv2.imread(img_file, cv2.IMREAD_COLOR)

dst = cv2.blur(src, (9,9), anchor=(-1, -1), borderType=cv2.BORDER_DEFAULT)


from matplotlib import pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10,5))
ax[0].imshow(src[:,:, ::-1]), ax[0].axis("off")
ax[1].imshow(dst[:,:, ::-1]), ax[1].axis("off")

plt.tight_layout()
plt.show()

### 윤곽선 검출

- 물체의 경계선이나 외곽선을 찾아내는 것

- 이미지는 픽셀 값으로 이뤄어져 있으므로, 윤곽선에서 픽셀 값이 급격히 변화함

- 대표적인 방법
  - Sobel 필터: 
    - 이미지의 가로방향, 세로방향 변화량을 계산함
    - 간단하고 빠르지만, 노이즈에 약할 수 있음
  - Laplacian 필터:
    - 밝기 변화의 2차 미분을 이용함. (경계가 있는 부분에서 값이 크게 변화)
    - 간단하지만, 노이즈에 약할 수 있음
  - Canny Edge Detection
    - 가장 널리 사용되는 알고리즘 중 하나
    - 다음의 순서로 동작함
    1. Gaussian Blur로 노이즈 제거
    2. Sobel로 gradient 계산
    3. Non-maximum suppression으로 엷은 경계만 남김
    4. Double trheshold로 강한/약한 경계 분류
    5. Hysteresis로 최종 경계 결정

In [ ]:
import cv2

img_file = "Lena.png"
src = cv2.imread(img_file, cv2.IMREAD_COLOR)

sobel     = cv2.Sobel(src, cv2.CV_8U, 1, 0, 3)
laplacian = cv2.Laplacian(src, cv2.CV_8U, ksize=3)
canny     = cv2.Canny(src, 100, 255)


from matplotlib import pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(12,3))
ax[0].imshow(src[:,:, ::-1]), ax[0].set_title("Source"), ax[0].axis("off")
ax[1].imshow(sobel, cmap="gray"), ax[1].set_title("Sobel"), ax[1].axis("off")
ax[2].imshow(laplacian, cmap="gray"), ax[2].set_title("Laplacian"), ax[2].axis("off")
ax[3].imshow(canny, cmap="gray"), ax[3].set_title("Canny"), ax[3].axis("off")

plt.tight_layout()
plt.show()

### 코너 검출

- 꼭지점은 추적하기 좋은 지점으로 널리 사용
- 다각형이나 객체의 꼭지점을 검출하는데 사용

![](https://drive.google.com/uc?export=view&id=1DHf5U3qxfK_ODqkgWerk8X72YAJIaVMj)

In [ ]:
# Chess Board
!wget 'https://drive.google.com/uc?export=download&id=1DHf5U3qxfK_ODqkgWerk8X72YAJIaVMj' -O Source.jpg

In [ ]:
# 2D Shapes
!wget 'https://drive.google.com/uc?export=download&id=14MHrwOPbWo3ZZA1ncaf7gIoH1wPFZUvx' -O Source.jpg

In [ ]:
import cv2
import numpy as np

# 입력 이미지 읽기
img_file = "Source.jpg"
src = cv2.imread(img_file, cv2.IMREAD_COLOR)

# 코너 검출 수행
gray = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY)
corners = cv2.goodFeaturesToTrack(gray, 100, 0.01, 5, blockSize=3, useHarrisDetector=True, k=0.03)
corners = np.int32(corners)

# 코너 검출 결과 이미지 생성
dst = src.copy()
for i in corners:
    cv2.circle(dst, tuple(i[0]), 3, (0, 0, 255), 4)


# 이미지 출력
from matplotlib import pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20,5))
ax[0].imshow(src[:,:, ::-1]), ax[0].set_title("Source"), ax[0].axis("off")
ax[1].imshow(dst[:,:, ::-1]), ax[1].set_title("Corners"), ax[1].axis("off")

plt.tight_layout()
plt.show()

### 직선 검출
- 허프 변환(Hough Transform)은 전통적인 영상처리 기반 직선 검출에 대표적인 방법 중 하나
- 직선 검출 목적에 따라 다른 방법들도 많이 사용됨
  - Canny + 윤곽선 분석
  - LSD, Line Segment Detector
  - 딥러닝 기반 방법

In [ ]:
# Road
!wget 'https://drive.google.com/uc?export=download&id=11jGBMY_m37r7Ib46kPRqksp0gwu3BvQB' -O Source.jpg

- Hough Transform
  
  ```
  cv2.HoughLines(image, rho, theta, threshold, srn=0, stn=0, main_theta=0, max_theta=np.pi)
  ```
  
  | 파라미터 | 설명 |
  | --- | --- |
  | image | 검출 이미지 (보통은 8-bit single-channel binary image 사용) |
  | rho | 거리 해상도 |
  | theta | 각도 해상도 |
  | threshold | 직선으로 인정하기 위한 누적 투표 수의 최소값 |


  - 선이 너무 많이 / 적게 검출된다 → threshold 증가 / 감소
  - 각도 정밀도가 부족하다 → theta 감소
  - 거리 정밀도가 부족하다 → rho 감소 

In [ ]:
import cv2
import numpy as np

# 입력 이미지 읽기
img_file = "Source.jpg"
src = cv2.imread(img_file, cv2.IMREAD_COLOR)

# 라인 검출 수행
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
canny = cv2.Canny(gray, 5000, 1500, apertureSize = 5, L2gradient = True)
lines = cv2.HoughLines(canny, 0.8, np.pi / 180, 150, srn = 100, stn = 200, min_theta = 0, max_theta = np.pi)

# 검출 결과 출력
dst = src.copy()
for i in lines:
    rho, theta = i[0][0], i[0][1]
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a*rho, b*rho

    scale = src.shape[0] + src.shape[1]

    x1 = int(x0 + scale * -b)
    y1 = int(y0 + scale * a)
    x2 = int(x0 - scale * -b)
    y2 = int(y0 - scale * a)

    cv2.line(dst, (x1, y1), (x2, y2), (0, 0, 255), 2)
    cv2.circle(dst, (int(x0), int(y0)), 3, (255, 0, 0), 5, cv2.FILLED)


# 이미지 출력
from matplotlib import pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20,5))
#ax[0].imshow(canny, cmap="gray"), ax[0].set_title("Canny"), ax[0].axis("off")
ax[0].imshow(src[:,:,::-1]), ax[0].set_title("Source"), ax[0].axis("off")
ax[1].imshow(dst[:,:,::-1]), ax[1].set_title("Lines"), ax[1].axis("off")

plt.tight_layout()
plt.show()

- Progressive Probabilistic Hough Transform

In [ ]:
import cv2
import numpy as np

# 입력 이미지 읽기
img_file = "Source.jpg"
src = cv2.imread(img_file, cv2.IMREAD_COLOR)

# 라인 검출 수행
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
canny = cv2.Canny(gray, 5000, 1500, apertureSize = 5, L2gradient = True)
lines = cv2.HoughLinesP(canny, 0.8, np.pi / 180, 90, minLineLength = 10, maxLineGap = 100)

# 검출 결과 출력
dst = src.copy()
for i in lines:
    cv2.line(dst, (int(i[0][0]), int(i[0][1])), (int(i[0][2]), int(i[0][3])), (0, 0, 255), 2)


# 이미지 출력
from matplotlib import pyplot as plt

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20,5))
ax[0].imshow(src[:,:,::-1]), ax[0].set_title("Source"), ax[0].axis("off")
ax[1].imshow(dst[:,:,::-1]), ax[1].set_title("Lines"), ax[1].axis("off")

plt.tight_layout()
plt.show()

## [실습과제]

인터넷에서 도로(자동차 없는)사진을 다운 받은 후, 라인검출을 통해 도로의 경계선을 표시하시오.

In [ ]:
# Write your code here...

----
## 참고 코드

### 이미지 업로드 방법
Colab의 작업 폴더('/content')에 이미지 파일을 업로드 한다. Google Drive를 마운트 해서 이미지 파일을 사용할 수도 있다. 이번 실습은 Colab 만으로 진행하려는 상황으로 (로컬 PC에서) 작업 이미지를 직접 업로드 하거나, wget 명령과 이미지 링크 정보를 이용하는 경우의 두 가지 방법을 진행한다.

![](https://drive.google.com/uc?export=view&id=1Ywls6xHfnvNEZ5Bxz9jUldMQDzUSqta4)

- 레나 이미지를 로컬 PC에 다운 받은 후 Colab에 업로드, 또는 공유 링크를 wget을 이용해서 직접 Colab으로 직접 다운로드 한다.

In [ ]:
# 로컬 PC에서 파일을 선택하여 업로드
from google.colab import files
uploaded = files.upload()

In [ ]:
# 알고 있는 링크 정보를 wget 명령을 통해서 Colab에 복사
!wget 'https://drive.google.com/uc?export=download&id=1Ywls6xHfnvNEZ5Bxz9jUldMQDzUSqta4' -O Lena.png

### 이미지 출력

- 이미지 객체를 생성 한다.

In [ ]:
import cv2
img = cv2.imread('Lena.png', cv2.IMREAD_COLOR)

- 코랩 환경에서 실행 할 때, `cv2_imshow` 모듈로 출력 한다.

In [ ]:
from google.colab.patches import cv2_imshow
cv2_imshow(img)